# Food Price Watch 03: price behaviour
A descriptive study of the 8 prices in `processed/data_8.csv`: how often they change, how much, how long they stay unchanged, and their sustained rise episodes (the reference that alerts are scored against in notebook 05). No model.
Rules: the 2022 gap is never bridged; a blank price is "not reported", never a zero change; real jumps are kept. Rises and pre-Ramadan changes are described, not explained.
Writes tables `reports/tables/B1`-`B6` (rates, counts, sizes of changes and dates; no price series), `processed/rise_episodes_8.csv` and two figures in `reports/figures/`.
Needs notebook 02 to have run. Run top to bottom and paste the last cell's output back.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Code comes fresh from GitHub main each session; Drive holds data and outputs only.
import os
if not os.path.isdir('/content/foodprice_watch/.git'):
    !git clone --single-branch --branch main https://github.com/mrchowdhury587/foodprice_watch.git /content/foodprice_watch
else:
    !git -C /content/foodprice_watch pull --ff-only origin main
%pip install -q -r /content/foodprice_watch/requirements-colab.txt

In [ ]:
import sys, os, json
os.chdir('/content/foodprice_watch')
sys.path.insert(0, '/content/foodprice_watch/src')
CODE_COMMIT = !git rev-parse --short HEAD
CODE_COMMIT = CODE_COMMIT[0]
from foodwatch.paths import SETTINGS, DATA_ROOT, RAW, data, raw, sha256, write_csv
for folder in ['raw', 'processed', 'reports/tables']:
    (DATA_ROOT / folder).mkdir(parents=True, exist_ok=True)
for tmp in DATA_ROOT.rglob('*.tmp'):   # half-written files from an interrupted session
    tmp.unlink()
print('code commit:', CODE_COMMIT, '| data root:', DATA_ROOT)

In [ ]:
import pandas as pd
from foodwatch import behaviour as bh
D = pd.read_csv(data('data'), parse_dates=['Date'])
h_data = sha256(data('data'))
r, b = SETTINGS['rises'], SETTINGS['behaviour']
P = bh.pairs(D)
B1 = bh.change_frequency(P)
B2 = bh.change_size(P)
B3 = bh.unchanged_runs(P)
B4 = bh.price_step(P)
Rd = bh.rise_days(D, r['window_reports'])
thresholds = sorted({r['threshold'], *r['sensitivity']})
E = {t: bh.episodes(Rd, t, r['persist_reports'], r['merge_gap_reports']) for t in thresholds}
B5 = bh.episodes_by_item_year(Rd, E, r['threshold'])
B6 = bh.ramadan(Rd, D, b['ramadan_window_days'], r['window_reports'], r['threshold'])
episodes = E[r['threshold']]

In [ ]:
# Checks: no change is counted across the 2022 gap or a blank price; no rise window touches the gap.
gap_idx = D.loc[D.gap_2022, 'report_idx']
first_after_gap = D.loc[(D.Date > pd.Timestamp(SETTINGS['gap_2022'][1])), 'report_idx'].min()
chk = dict(
    changes_counted=len(P),
    change_inside_or_across_gap=int(P.report_idx.isin(gap_idx).sum() + (P.report_idx == first_after_gap).sum()),
    rise_window_touching_gap=int(Rd.R.notna()[(Rd.report_idx >= gap_idx.min()) & (Rd.report_idx - r['window_reports'] <= gap_idx.max())].sum()),
    episodes_10pct=len(episodes),
    episode_onsets_in_gap=int(episodes.onset_idx.isin(gap_idx).sum()),
)
chk['ok'] = chk['change_inside_or_across_gap'] == 0 and chk['rise_window_touching_gap'] == 0 and chk['episode_onsets_in_gap'] == 0
assert chk['ok'], chk

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
FIG = DATA_ROOT / SETTINGS['files']['figures']; FIG.mkdir(parents=True, exist_ok=True)
short = lambda s: s.split(' | ')[0]

# F1: share of report days with a price change, item x year (a rate, no prices)
H = B1[B1.year != 'all'].pivot(index='item_id', columns='year', values='share_changed')
fig, ax = plt.subplots(figsize=(7.5, 4))
im = ax.imshow(H.values, cmap='Blues', vmin=0, vmax=0.6, aspect='auto')
# year labels show the months covered: 2022 counts only the days before the gap
used = D[~D.gap_2022 & (D.Date < pd.Timestamp(SETTINGS['gap_2022'][0])) | (D.Date > pd.Timestamp(SETTINGS['gap_2022'][1]))]
span = used.groupby(used.Date.dt.year).Date.agg(['min', 'max'])
full = lambda y: span.loc[y, 'min'].month == 1 and span.loc[y, 'max'].month == 12
lab = [y if full(int(y)) else y + chr(10) + span.loc[int(y), 'min'].strftime('%b') + '-' + span.loc[int(y), 'max'].strftime('%b') for y in H.columns]
ax.set_xticks(range(H.shape[1]), lab); ax.set_yticks(range(H.shape[0]), [short(i) for i in H.index])
for (i, j), v in pd.DataFrame(H.values).stack().items():
    ax.text(j, i, f'{v:.2f}', ha='center', va='center', fontsize=8, color='white' if v > 0.4 else '#1a1a1a')
ax.set_title('Share of report days on which the price changed', loc='left', fontsize=11)
for s in ax.spines.values(): s.set_visible(False)
fig.colorbar(im, ax=ax, shrink=0.8)
fig.tight_layout(); fig.savefig(FIG / 'F1_change_frequency.png', dpi=160); plt.close(fig)

# F2: price series with sustained rise episodes shaded (contains prices: stays on Drive)
items = list(H.index)
fig, axes = plt.subplots(4, 2, figsize=(11, 10), sharex=True)
g0, g1 = pd.Timestamp(SETTINGS['gap_2022'][0]), pd.Timestamp(SETTINGS['gap_2022'][1])
for ax, item in zip(axes.flat, items):
    x = D[(D.item_id == item) & ~D.gap_2022]
    pre, post = x[x.Date < g0], x[x.Date > g1]
    for part in (pre, post):
        ax.plot(part.Date, part.Price_Avg, color='#2a6fb0', lw=1.2)
    ax.axvspan(g0, g1, color='#d9d9d9', alpha=0.6, lw=0)
    for e in episodes[episodes.item_id == item].itertuples():
        ax.axvspan(e.onset_date, e.end_date, color='#e07b39', alpha=0.35, lw=0)
    ax.set_title(item, loc='left', fontsize=9)
    ax.tick_params(labelsize=8)
    for s in ('top', 'right'): ax.spines[s].set_visible(False)
fig.suptitle('Daily price (Tk) with sustained 10 % rise episodes (orange); 2022 gap in grey', x=0.01, ha='left', fontsize=11)
fig.tight_layout(); fig.savefig(FIG / 'F2_prices_with_rises.png', dpi=160); plt.close(fig)

In [ ]:
T = DATA_ROOT / SETTINGS['files']['tables']
for name, df in [('B1_change_frequency', B1), ('B2_change_size', B2), ('B3_unchanged_runs', B3), ('B4_price_step', B4),
                 ('B5_rise_episodes_by_item_year', B5), ('B6_ramadan', B6)]:
    write_csv(df, T / f'{name}.csv')
h_eps = write_csv(episodes.assign(onset_date=episodes.onset_date.dt.date, end_date=episodes.end_date.dt.date), data('rise_episodes'))

In [ ]:
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 30)
P2 = lambda df: df.assign(item_id=df.item_id.map(short)).to_string(index=False)
yr = lambda y: B5.year.between(*y)
print('=== 03 price behaviour summary ===')
print('code commit:', CODE_COMMIT, '| data_8.csv sha256', h_data[:16])
print('checks:', 'PASS' if chk['ok'] else 'FAIL', {k: v for k, v in chk.items() if k != 'ok'})
print('rise rule: R over', r['window_reports'], 'report days >=', r['threshold'], 'for >=', r['persist_reports'], 'report days; merge gap', r['merge_gap_reports'])
print('\nB1 share of report days with a change, by year (2021 from 17 May; 2022 = 1 Jan to 11 Mar, before the gap; 2026 to 24 Oct)')
print(P2(B1.pivot(index='item_id', columns='year', values='share_changed').reset_index()))
print('\nB2 size of a change (on days it changed)'); print(P2(B2))
print('\nB3 unchanged runs (report days)'); print(P2(B3))
print('\nB4 price step (Tk)'); print(P2(B4))
print('\nB5 sustained rise episodes by onset year (10 % rule)')
print(P2(B5.pivot(index='item_id', columns='year', values='episodes_10pct').fillna(0).astype(int).reset_index()))
for lab, y in [('validation 2024', (2024, 2024)), ('retrospective 2025-26', (2025, 2026))]:
    s = B5[yr(y)]
    print(f'{lab}: episodes 7 % / 10 % / 15 % =', int(s.episodes_7pct.sum()), '/', int(s.episodes_10pct.sum()), '/', int(s.episodes_15pct.sum()),
          '| items with no 10 % episode:', sorted(short(i) for i in set(D.item_id) - set(s.loc[s.episodes_10pct > 0, 'item_id'])))
print('\nB6 before Ramadan vs other days (descriptive;', B6.attrs['window'] + ')'); print(P2(B6))
print('\nrise episodes file:', SETTINGS['files']['rise_episodes'], '| rows', len(episodes), '| sha256', h_eps[:16])
print('figures:', sorted(p.name for p in FIG.glob('*.png')))